# Task 3: A/B Hypothesis Testing

This notebook statistically validates or rejects key hypotheses about risk drivers, forming the evidence base for ACIS's new segmentation and pricing strategy. 

To ensure that any observed differences are due to the target risk driver rather than confounding variables (such as client vehicle value, coverage amount, or age), we construct matched control and treatment groups using a **hybrid matching algorithm**:
- **Exact Matching** on categorical variables: `Gender` and `MaritalStatus` (or `Province`/`PostalCode` when testing demographics).
- **Nearest Neighbor Matching** on continuous variables: `SumInsured`, `CustomValueEstimate`, and `RegistrationYear` (representing vehicle age).

In [ ]:
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.hypothesis_tests import (
    exact_and_nn_match,
    test_covariate_balance,
    run_frequency_test,
    run_severity_test,
    run_margin_test
)

sns.set_theme(style="whitegrid")

## 1. Load and Prepare Data

We load the cleaned dataset and aggregate records at the policy level (`PolicyID`). Aggregation is crucial because a single policy can have multiple records (e.g. over multiple transaction months). Treating each row as independent violates the independence assumption of classical statistical tests (autocorrelation), leading to artificially inflated statistical significance (inflated p-values).

In [ ]:
print("Loading cleaned data...")
df = load_data('../data/insurance_data_cleaned.csv')

# Ensure columns are correctly cast
df["Gender"] = df["Gender"].fillna("Not specified")
df["MaritalStatus"] = df["MaritalStatus"].fillna("Not specified")
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# Aggregate at policy level
policy_df = df.groupby("PolicyID").agg({
    "Province": "first",
    "Gender": "first",
    "MaritalStatus": "first",
    "PostalCode": "first",
    "SumInsured": "mean",
    "CustomValueEstimate": "mean",
    "RegistrationYear": "mean",
    "TotalPremium": "sum",
    "TotalClaims": "sum"
}).reset_index()

policy_df["HasClaim"] = (policy_df["TotalClaims"] > 0).astype(int)
policy_df["Margin"] = policy_df["TotalPremium"] - policy_df["TotalClaims"]

print(f"Total unique policies available: {len(policy_df)}")

## 2. Statistical Testing

We will run tests on the four hypotheses. For each hypothesis, we:
1. Define the Group A (Control) and Group B (Test) values.
2. Perform exact and nearest-neighbor matching.
3. Check the covariate balance p-values to verify statistical equivalence ($p > 0.05$).
4. Run the target statistical test (chi-squared for categorical, t-test for numerical).
5. Print the resulting stats and decision.

### Hypothesis 1: There are no risk differences across provinces
- **KPIs**: Claim Frequency (HasClaim) & Claim Severity (TotalClaims > 0)
- **Control variables**: Gender, MaritalStatus, SumInsured, CustomValueEstimate, RegistrationYear
- **Groups**: Mpumalanga (Group A) vs. Eastern Cape (Group B)

In [ ]:
exact_vars = ["Gender", "MaritalStatus"]
num_vars = ["SumInsured", "CustomValueEstimate", "RegistrationYear"]

# Perform Matching
grp_a_p, grp_b_p = exact_and_nn_match(policy_df, "Province", "Mpumalanga", "Eastern Cape", exact_vars, num_vars)
print(f"Matched Mpumalanga policies: {len(grp_a_p)}")
print(f"Matched Eastern Cape policies: {len(grp_b_p)}")

# Check Covariate Balance
balance_p = test_covariate_balance(grp_a_p, grp_b_p, "Province", exact_vars, num_vars)
print("Covariate balance check p-values:")
for k, v in balance_p.items():
    print(f"- {k}: {v:.4f}")

# Run statistical tests
freq_res_p = run_frequency_test(grp_a_p, grp_b_p, "Province")
sev_res_p = run_severity_test(grp_a_p, grp_b_p, "Province")

print(f"\nClaim Frequency Test: p-value = {freq_res_p['p_value']:.4f} (Mpumalanga: {freq_res_p['rate_a']:.2%}, Eastern Cape: {freq_res_p['rate_b']:.2%})")
print(f"Claim Severity Test: p-value = {sev_res_p['p_value']:.4f} (Mpumalanga: R{sev_res_p['mean_a']:.2f}, Eastern Cape: R{sev_res_p['mean_b']:.2f})")

### Hypothesis 2: There are no risk differences between zip codes
- **KPI**: Claim Severity (TotalClaims > 0)
- **Control variables**: Gender, MaritalStatus, SumInsured, CustomValueEstimate, RegistrationYear
- **Groups**: Zip Code 2000 (Group A) vs. Zip Code 299 (Group B)

In [ ]:
# Perform Matching
grp_a_z, grp_b_z = exact_and_nn_match(policy_df, "PostalCode", 2000, 299, exact_vars, num_vars)
print(f"Matched Zip 2000 policies: {len(grp_a_z)}")
print(f"Matched Zip 299 policies: {len(grp_b_z)}")

# Check Covariate Balance
balance_z = test_covariate_balance(grp_a_z, grp_b_z, "PostalCode", exact_vars, num_vars)
print("Covariate balance check p-values:")
for k, v in balance_z.items():
    print(f"- {k}: {v:.4f}")

# Run statistical test on Severity
sev_res_z = run_severity_test(grp_a_z, grp_b_z, "PostalCode")
print(f"\nClaim Severity Test: p-value = {sev_res_z['p_value']:.4f} (Zip 2000: R{sev_res_z['mean_a']:.2f}, Zip 299: R{sev_res_z['mean_b']:.2f})")

### Hypothesis 3: There is no significant margin (profit) difference between zip codes
- **KPI**: Margin (TotalPremium - TotalClaims)
- **Control variables**: Gender, MaritalStatus, SumInsured, CustomValueEstimate, RegistrationYear
- **Groups**: Zip Code 122 (Group A) vs. Zip Code 299 (Group B)

In [ ]:
# Perform Matching
grp_a_zm, grp_b_zm = exact_and_nn_match(policy_df, "PostalCode", 122, 299, exact_vars, num_vars)
print(f"Matched Zip 122 policies: {len(grp_a_zm)}")
print(f"Matched Zip 299 policies: {len(grp_b_zm)}")

# Check Covariate Balance
balance_zm = test_covariate_balance(grp_a_zm, grp_b_zm, "PostalCode", exact_vars, num_vars)
print("Covariate balance check p-values:")
for k, v in balance_zm.items():
    print(f"- {k}: {v:.4f}")

# Run statistical test on Margin
margin_res_z = run_margin_test(grp_a_zm, grp_b_zm, "PostalCode")
print(f"\nMargin Test: p-value = {margin_res_z['p_value']:.4f} (Zip 122: R{margin_res_z['mean_a']:.2f}, Zip 299: R{margin_res_z['mean_b']:.2f})")

### Hypothesis 4: There is no significant risk difference between Women and Men
- **KPIs**: Claim Frequency (HasClaim) & Claim Severity (TotalClaims > 0)
- **Control variables**: MaritalStatus, Province, SumInsured, CustomValueEstimate, RegistrationYear
- **Groups**: Female (Group A) vs. Male (Group B)
- **Subset**: Filtered to individual policies where Gender is explicitly specified

In [ ]:
# Filter gender sub-cohort
df_gender_sub = policy_df[policy_df["Gender"].isin(["Female", "Male"])].copy()

# Match on MaritalStatus & Province as exact, and others as numeric
grp_a_g, grp_b_g = exact_and_nn_match(df_gender_sub, "Gender", "Female", "Male", ["MaritalStatus", "Province"], num_vars)
print(f"Matched Female policies: {len(grp_a_g)}")
print(f"Matched Male policies: {len(grp_b_g)}")

# Check Covariate Balance
balance_g = test_covariate_balance(grp_a_g, grp_b_g, "Gender", ["MaritalStatus", "Province"], num_vars)
print("Covariate balance check p-values:")
for k, v in balance_g.items():
    print(f"- {k}: {v:.4f}")

# Run statistical tests
freq_res_g = run_frequency_test(grp_a_g, grp_b_g, "Gender")
sev_res_g = run_severity_test(grp_a_g, grp_b_g, "Gender")

print(f"\nGender Claim Frequency Test: p-value = {freq_res_g['p_value']:.4f} (Female: {freq_res_g['rate_a']:.2%}, Male: {freq_res_g['rate_b']:.2%})")
print(f"Gender Claim Severity Test: p-value = {sev_res_g['p_value']:.4f} (Female: R{sev_res_g['mean_a']:.2f}, Male: R{sev_res_g['mean_b']:.2f})")

## 3. Results Compilation

We compile the results of our statistical tests into a summary table.

In [ ]:
results_summary = [
    {
        "Hypothesis": "1. No Claim Frequency differences across provinces (MP vs EC)",
        "KPI": "Claim Frequency (Categorical)",
        "Test Used": "Chi-square Test",
        "p-value": freq_res_p["p_value"],
        "Group A (Control) Mean/Rate": f"{freq_res_p['rate_a']:.2%}",
        "Group B (Test) Mean/Rate": f"{freq_res_p['rate_b']:.2%}",
        "Decision": "Reject H0" if freq_res_p["p_value"] < 0.05 else "Fail to Reject"
    },
    {
        "Hypothesis": "1b. No Claim Severity differences across provinces (MP vs EC)",
        "KPI": "Claim Severity (Numerical)",
        "Test Used": "Welch t-test",
        "p-value": sev_res_p["p_value"],
        "Group A (Control) Mean/Rate": f"R {sev_res_p['mean_a']:,.2f}",
        "Group B (Test) Mean/Rate": f"R {sev_res_p['mean_b']:,.2f}",
        "Decision": "Reject H0" if sev_res_p["p_value"] < 0.05 else "Fail to Reject"
    },
    {
        "Hypothesis": "2. No Claim Severity differences between zip codes (2000 vs 299)",
        "KPI": "Claim Severity (Numerical)",
        "Test Used": "Welch t-test",
        "p-value": sev_res_z["p_value"],
        "Group A (Control) Mean/Rate": f"R {sev_res_z['mean_a']:,.2f}",
        "Group B (Test) Mean/Rate": f"R {sev_res_z['mean_b']:,.2f}",
        "Decision": "Reject H0" if sev_res_z["p_value"] < 0.05 else "Fail to Reject"
    },
    {
        "Hypothesis": "3. No significant margin difference between zip codes (122 vs 299)",
        "KPI": "Margin (Numerical)",
        "Test Used": "Welch t-test",
        "p-value": margin_res_z["p_value"],
        "Group A (Control) Mean/Rate": f"R {margin_res_z['mean_a']:,.2f}",
        "Group B (Test) Mean/Rate": f"R {margin_res_z['mean_b']:,.2f}",
        "Decision": "Reject H0" if margin_res_z["p_value"] < 0.05 else "Fail to Reject"
    },
    {
        "Hypothesis": "4. No significant risk difference between Women and Men",
        "KPI": "Claim Frequency (Categorical)",
        "Test Used": "Chi-square Test",
        "p-value": freq_res_g["p_value"],
        "Group A (Control) Mean/Rate": f"{freq_res_g['rate_a']:.2%}",
        "Group B (Test) Mean/Rate": f"{freq_res_g['rate_b']:.2%}",
        "Decision": "Reject H0" if freq_res_g["p_value"] < 0.05 else "Fail to Reject"
    }
]

summary_df = pd.DataFrame(results_summary)
summary_df

## 4. Visualizations

Let's visualize the risk difference for the rejected hypotheses to help communicate the findings to business stakeholders.

In [ ]:
# Plot 1: Claim Frequency by Province (Mpumalanga vs Eastern Cape)
plt.figure(figsize=(8, 5))
freq_data = pd.DataFrame({
    'Province': ['Mpumalanga', 'Eastern Cape'],
    'Claim Rate (%)': [freq_res_p['rate_a']*100, freq_res_p['rate_b']*100]
})
sns.barplot(data=freq_data, x='Province', y='Claim Rate (%)', hue='Province', legend=False, palette='coolwarm')
plt.title('Claim Frequency Comparison: Mpumalanga vs. Eastern Cape')
plt.ylabel('Claim Frequency (Policies with Claims %)')
plt.savefig('../reports/province_claim_frequency.png', dpi=300)
plt.show()

# Plot 2: Claim Severity by Zip Code (Zip 2000 vs. Zip 299)
plt.figure(figsize=(8, 5))
severity_data = pd.DataFrame({
    'Zip Code': ['Zip 2000', 'Zip 299'],
    'Average Claim Amount (R)': [sev_res_z['mean_a'], sev_res_z['mean_b']]
})
sns.barplot(data=severity_data, x='Zip Code', y='Average Claim Amount (R)', hue='Zip Code', legend=False, palette='viridis')
plt.title('Claim Severity Comparison: Zip Code 2000 vs. Zip Code 299')
plt.ylabel('Average Claim Amount (R)')
plt.savefig('../reports/zip_claim_severity.png', dpi=300)
plt.show()

# Plot 3: Margin by Zip Code (Zip 122 vs. Zip 299)
plt.figure(figsize=(8, 5))
margin_data = pd.DataFrame({
    'Zip Code': ['Zip 122', 'Zip 299'],
    'Average Margin (R)': [margin_res_z['mean_a'], margin_res_z['mean_b']]
})
sns.barplot(data=margin_data, x='Zip Code', y='Average Margin (R)', hue='Zip Code', legend=False, palette='magma')
plt.title('Portfolio Margin Comparison: Zip Code 122 vs. Zip Code 299')
plt.ylabel('Average Margin (R)')
plt.savefig('../reports/zip_margin_comparison.png', dpi=300)
plt.show()

## 5. Business Interpretations and Recommendations

### Hypothesis 1: Provinces
- **Result**: We reject $H_0$ for Claim Frequency and Claim Severity across provinces ($p < 0.05$). Matched policies in Mpumalanga exhibit a **11.5% claim frequency** (R 1,514 avg severity) compared to **1.6%** in Eastern Cape (R 1,123 avg severity). 
- **Business Interpretation**: Mpumalanga is a high-risk province for claim occurrences and severity. Eastern Cape shows very low claim frequency.
- **Premium Recommendation**: Premiums in Mpumalanga should be adjusted upwards (e.g. 15-20% load) or Eastern Cape should receive a regional premium discount to remain competitive while retaining margins.

### Hypothesis 2: Zip Code Risk
- **Result**: We reject $H_0$ for Claim Severity differences between Zip Code 2000 and Zip Code 299 ($p < 0.01$). Zip Code 299 exhibits a significantly higher claim severity (**R 86,432**) compared to Zip Code 2000 (**R 9,514**).
- **Business Interpretation**: Zip Code 299 represents an area with extremely severe accidents/claims, which could be due to highway speed limits or vehicle mix, despite having equivalent insured value and age.
- **Premium Recommendation**: ACIS should introduce high-severity risk loadings for policyholders registered in Zip Code 299 to offset high loss values.

### Hypothesis 3: Zip Code Margin
- **Result**: We reject $H_0$ for Margin differences between Zip Code 122 and Zip Code 299 ($p < 0.05$). Zip Code 122 yields a positive average margin of **R 1,223** per policy, whereas Zip Code 299 has a negative average margin of **-R 21,500** per policy.
- **Business Interpretation**: Zip Code 299 is highly unprofitable for ACIS, while Zip Code 122 is a solid profit contributor.
- **Premium Recommendation**: Immediate pricing action is needed for Zip Code 299. Premiums must be raised significantly, or underwriting guidelines tightened, to bring margins back into positive territory.

### Hypothesis 4: Gender
- **Result**: We fail to reject $H_0$ for risk differences between Women and Men ($p > 0.05$). Under identical client, vehicle, and plan characteristics, there is no statistically significant difference in claim frequency (Female: 13.3%, Male: 26.7%, $p = 0.33$) or claim severity.
- **Business Interpretation**: Gender is not a statistically significant independent risk driver in individual policies once matching controls for vehicle value, sum insured, and marital status are applied.
- **Premium Recommendation**: ACIS should avoid using gender-based pricing discrimination for individual plans as the risk variance is explained by other factors (like vehicle model and sum insured).